# Mizan · Export per-example test predictions

Loads the **published** fine-tuned model from Hugging Face, predicts every test query, and saves
`results/arabert_test_predictions.csv` (text, label, pred, score, variety).

The Results page uses this file to show per-intent F1, the most common confusions, accuracy by score band,
and real successes and failures. No retraining is needed, and a CPU runtime is fine (about 1-2 minutes).

In [ ]:
REPO_URL = "https://github.com/zainabali18/Mizan.git"
!git clone -q $REPO_URL mizan_repo
%cd mizan_repo
!pip install -q -r requirements.txt transformers
!python -m mizan.data > /dev/null && echo "processed data ready"

In [ ]:
import json, numpy as np, pandas as pd
from mizan.service import MizanClassifier

clf = MizanClassifier()                     # reads mizan_config.json -> the published model
test = pd.read_csv("data/processed/test.csv")
rows = []
for t, lab, var in zip(test.text, test.label, test.variety):
    r = clf.classify(t)
    rows.append({"text": t, "label": lab, "pred": r["intent"], "score": r["confidence"], "variety": var})
pred = pd.DataFrame(rows)
print("accuracy:", (pred.label == pred.pred).mean().round(4), "| rows:", len(pred))

The accuracy should match the recorded result (0.7919). A small difference can come from running on CPU instead of GPU.

In [ ]:
from sklearn.metrics import f1_score
print("macro-F1:", round(f1_score(pred.label, pred.pred, average="macro", zero_division=0), 4))
pred.to_csv("results/arabert_test_predictions.csv", index=False)
from google.colab import files
files.download("results/arabert_test_predictions.csv")   # -> put in results/ in your repository